# Kaggle Adapter Training Notebook

Trains a `train + val` adapter run, saves per-epoch adapter checkpoints to `/kaggle/working`, and saves a final `adapter_run8/` folder for export.

**Run 8 config:** `r=8`, `alpha=32`, `grad_accum=4`, attention+MLP LoRA, flat `2e-4`, `3` epochs, `MAX_CONTEXT_CHARS=400`.

**After this notebook finishes:** create a Kaggle dataset from the output adapter folder, then attach that dataset in `run8_inference_kaggle.ipynb`.


## 0. Install libraries

Uses the current project starter versions.


In [ ]:
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow


## 1. Imports and configuration

This follows the current starter notebook logic: auto-detect Kaggle input paths, prefer a local attached SmolVLM dataset when present, and write outputs to `/kaggle/working`.


In [ ]:
import glob
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

_kaggle_input = Path('/kaggle/input')
if _kaggle_input.exists():
    print('Contents of /kaggle/input:')
    for _entry in sorted(_kaggle_input.rglob('*'))[:40]:
        print(f'  {_entry}')

    _matches = glob.glob('/kaggle/input/**/train.csv', recursive=True)
    print(f'\nFound train.csv at: {_matches}')
    if not _matches:
        raise RuntimeError(
            'train.csv not found under /kaggle/input. '
            'Make sure the competition dataset is attached to this notebook.'
        )
    DATA_DIR = Path(_matches[0]).parent
    print(f'DATA_DIR set to: {DATA_DIR}')
else:
    DATA_DIR = Path('data')

_HF_MODEL_ID = 'HuggingFaceTB/SmolVLM-500M-Instruct'
_local_model = Path('/kaggle/input/smolvlm-500m-instruct')
MODEL_ID = str(_local_model) if _local_model.exists() else _HF_MODEL_ID
print(f'Model source: {MODEL_ID}')

WORKING_DIR = Path('/kaggle/working')
ADAPTER_NAME = 'adapter_run8'
ADAPTER_DIR = WORKING_DIR / ADAPTER_NAME
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)

LORA_R = 8
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LR = 2e-4
NUM_EPOCHS = 3
GRAD_ACCUM_STEPS = 4
IMG_SIZE = 224
MAX_CONTEXT_CHARS = 400
LOG_EVERY = 50
CHOICE_LETTERS = 'ABCDEFGHIJ'


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\nUsing device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 2. Load data

This keeps the Run 5 / Run 8 full-data recipe by concatenating `train` and `val` for training.


In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df = pd.read_csv(DATA_DIR / 'val.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')

for df in [train_df, val_df, test_df]:
    df['choices'] = df['choices'].apply(json.loads)

combined_df = pd.concat([train_df, val_df], ignore_index=True)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print(f'Combined training set: {len(combined_df):,} examples')


## 3. Prompt builder and dataset

Matches the starter notebook prompt and image-resolution behavior. It caps `lecture` and `hint` before the processor sees them, so there is no `truncation=True` issue.


In [ ]:
def _resolve_image_path(data_dir: Path, rel_path: str) -> Path:
    parts = Path(rel_path).parts
    candidates = [
        data_dir / rel_path,
        data_dir / 'images' / rel_path,
        data_dir / Path(rel_path).name,
        data_dir / 'images' / Path(rel_path).name,
    ]
    if parts[0] == 'images' and len(parts) > 1:
        candidates.append(data_dir / Path(*parts[1:]))
        candidates.append(data_dir / 'images' / Path(*parts[1:]))

    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError('Image not found. Tried:\n' + '\n'.join(f'  {p}' for p in candidates))


def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    context_parts = []
    lecture = row.get('lecture', '')
    hint = row.get('hint', '')
    if pd.notna(lecture) and str(lecture).strip():
        ctx = str(lecture).strip()[:MAX_CONTEXT_CHARS]
        context_parts.append(ctx)
    if pd.notna(hint) and str(hint).strip():
        ctx = str(hint).strip()[:MAX_CONTEXT_CHARS]
        context_parts.append(ctx)
    context_str = '\n'.join(context_parts)

    choices_str = '\n'.join(
        f'  {CHOICE_LETTERS[i]}. {c}' for i, c in enumerate(row['choices'])
    )

    prompt = '<image>\n'
    if context_str:
        prompt += f'Context:\n{context_str}\n\n'
    prompt += f'Question: {row["question"]}\n'
    prompt += f'Choices:\n{choices_str}\n'
    prompt += 'Answer:'

    if include_answer:
        prompt += f' {CHOICE_LETTERS[int(row["answer"])]}'

    return prompt


class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = 224):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = Image.open(_resolve_image_path(self.data_dir, row['image_path'])).convert('RGB')
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        return {
            'image': img,
            'text': build_prompt(row, include_answer=True),
            'answer': int(row['answer']),
        }

print(_resolve_image_path(DATA_DIR, train_df.iloc[0]['image_path']))
print(build_prompt(train_df.iloc[0], include_answer=True))


## 4. Load processor


In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
print('Processor loaded.')


## 5. Load model and attach Run 8 LoRA

This keeps the Run 5 attention+MLP target modules and changes only `alpha=32` and `grad_accum=4`.


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    low_cpu_mem_usage=True,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
assert n_trainable < 5_000_000, f'OVER 5M CAP: {n_trainable:,} params!'
print(f'Trainable params: {n_trainable:,} ? under 5M cap OK')


## 6. DataLoader and collate

Like the starter notebook, this avoids processor-level truncation and computes loss only on the final answer letter token.


In [ ]:
def collate_train(batch):
    inputs = processor(
        text=[item['text'] for item in batch],
        images=[item['image'] for item in batch],
        return_tensors='pt',
    )
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    labels = inputs['input_ids'].clone()
    labels[:, :-1] = -100
    inputs['labels'] = labels
    return inputs

train_ds = ScienceQADataset(combined_df, DATA_DIR, img_size=IMG_SIZE)
train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=collate_train,
    num_workers=0,
)
print(f'Training on {len(combined_df):,} examples ({len(train_df):,} train + {len(val_df):,} val)')
print(f'Batches per epoch: {len(train_loader):,}')
print(f'Effective batch size: {GRAD_ACCUM_STEPS} (grad_accum={GRAD_ACCUM_STEPS})')


## 7. Save helper

Saves a checkpoint after every epoch so a timeout does not lose everything.


In [ ]:
def save_adapter(epoch_label: str):
    ckpt_dir = ADAPTER_DIR.parent / f'{ADAPTER_NAME}_{epoch_label}'
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(ckpt_dir)
    processor.save_pretrained(ckpt_dir)
    total_mb = sum(f.stat().st_size for f in ckpt_dir.iterdir() if f.is_file()) / 1e6
    print(f'  Saved adapter ({epoch_label}) -> {ckpt_dir} ({total_mb:.1f} MB)')

print('Save helper defined.')


## 8. Train Run 8

This includes the fixed final gradient-accumulation step so the last partial batch of each epoch is not dropped.


In [ ]:
def train_run8(model, train_loader, num_epochs=NUM_EPOCHS, lr=LR, grad_accum=GRAD_ACCUM_STEPS):
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=0.01,
    )
    use_amp = torch.cuda.is_available()
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    total_steps = num_epochs * len(train_loader)

    print(f'\n{"=" * 60}')
    print('Run 8 Training Config')
    print(f'  Epochs:         {num_epochs}')
    print(f'  Steps/epoch:    {len(train_loader):,}')
    print(f'  Total steps:    {total_steps:,}')
    print(f'  LR:             {lr} (flat)')
    print(f'  Grad accum:     {grad_accum}')
    print(f'  LoRA alpha:     {LORA_ALPHA}')
    print(f'  Adapter dir:    {ADAPTER_DIR}')
    print(f'  AMP:            {use_amp}')
    print(f'{"=" * 60}\n')

    train_start = time.time()

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0.0
        epoch_start = time.time()
        optimizer.zero_grad()
        print(f'--- Epoch {epoch + 1}/{num_epochs} ---')

        for step, batch in enumerate(train_loader):
            with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
                loss = model(**batch).loss / grad_accum

            scaler.scale(loss).backward()
            total_loss += loss.item() * grad_accum

            if (step + 1) % grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            if (step + 1) % LOG_EVERY == 0 or step == 0:
                avg = total_loss / (step + 1)
                elapsed = time.time() - epoch_start
                sps = elapsed / (step + 1)
                eta_epoch = (len(train_loader) - step - 1) * sps
                global_step = epoch * len(train_loader) + step + 1
                eta_total = (total_steps - global_step) * ((time.time() - train_start) / global_step)
                print(
                    f'  [E{epoch+1} {step+1:>5}/{len(train_loader)}] '
                    f'loss={avg:.4f} | {sps:.2f}s/step | '
                    f'epoch ETA {eta_epoch/60:.1f}min | total ETA {eta_total/60:.0f}min'
                )

        if len(train_loader) % grad_accum != 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        avg_loss = total_loss / len(train_loader)
        epoch_mins = (time.time() - epoch_start) / 60
        print(f'\nEpoch {epoch + 1} done in {epoch_mins:.1f}min | avg loss: {avg_loss:.4f}')
        save_adapter(f'epoch{epoch+1}')

    print(f'\nTraining complete in {(time.time() - train_start)/60:.1f}min')
    return model

model = train_run8(model, train_loader)


## 9. Save final adapter

This writes the final folder you should export from Kaggle and attach to the inference notebook later.


In [ ]:
model.save_pretrained(ADAPTER_DIR)
processor.save_pretrained(ADAPTER_DIR)

total_mb = sum(f.stat().st_size for f in ADAPTER_DIR.iterdir() if f.is_file()) / 1e6
print(f'Final adapter saved -> {ADAPTER_DIR}')
print(f'Files: {len(list(ADAPTER_DIR.iterdir()))} | Size: {total_mb:.1f} MB')
for f in sorted(ADAPTER_DIR.iterdir()):
    print(f'  {f.name}')

print('\nTraining notebook complete.')
print('Create a Kaggle dataset from /kaggle/working/adapter_run8 and attach it in generate_submission_kaggle.ipynb.')
